# 10 完整评分卡建模流程

基于 `hscredit.xlsx` 真实信贷数据 (12448条样本, 83个特征)，
从原始数据到评分卡产出，覆盖完整建模流程。

**流程概览**
```
EDA → 特征筛选 → 最优分箱 → WOE编码 → LR建模 → 评估 → 评分卡 → 稳定性监控
```

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import hscredit as hsc
from hscredit import (
    init_setting, seed_everything,
    OptimalBinning, WOEEncoder, LogisticRegression, ScoreCard,
    NullSelector, VarianceSelector, IVSelector, CorrSelector, PSISelector,
    CompositeFeatureSelector,
    bin_plot, ks_plot, plot_weights, score_dist_plot, score_bin_plot,
    score_ks_plot, score_approval_badrate_curve, variable_iv_plot,
    threshold_analysis_plot,
)
from hscredit.core.metrics import ks, auc, gini, batch_psi, psi
from hscredit.core.metrics.stability import psi_rating
from hscredit.core.models.probability_to_score import probability_to_score
from hscredit import (batch_iv_analysis, missing_analysis, data_quality_report)

seed_everything(42); init_setting()

# ── 加载数据 ──────────────────────────────────────────
df = pd.read_excel('hscredit.xlsx')
df['loan_date'] = pd.to_datetime(df['loan_date'], unit='D', origin='1899-12-30')
df['loan_month'] = df['loan_date'].dt.to_period('M').astype(str)
target = 'FPD15'
exclude = ['MOB1','MOB2','loan_date','loan_month','FPD15','SFPD15']
all_feats = [c for c in df.columns if c not in exclude]
print(f'样本数: {len(df):}  坏率: {df[target].mean():.2%}  特征数: {len(all_feats)}')

## Step 1. EDA — 数据探索

In [ ]:
print('=== 缺失率TOP10 ===')
print(missing_analysis(df[all_feats]).sort_values('缺失率',ascending=False).head(10))
print('\n=== 数据质量 ===')
print(data_quality_report(df[all_feats]).head(5))

In [ ]:
print('=== 批量IV分析 ===')
df_filled = df[all_feats].fillna(-999)
batch_iv = batch_iv_analysis(df_filled.assign(**{target: df[target]}), features=all_feats, target=target)
print(batch_iv.sort_values('IV值', ascending=False).head(15))

## Step 2. 训练集/测试集划分（按时间）

In [ ]:
# 按时间顺序划分，避免数据穿越
df_sorted = df.sort_values('loan_date').reset_index(drop=True)
cutoff = int(len(df_sorted) * 0.7)
df_tr = df_sorted.iloc[:cutoff].copy()
df_te = df_sorted.iloc[cutoff:].copy()

X_tr_raw = df_tr[all_feats].fillna(-999)
X_te_raw = df_te[all_feats].fillna(-999)
y_tr = df_tr[target]; y_te = df_te[target]

print(f'训练集: {len(df_tr):} 条, 坏率: {y_tr.mean():.2%}')
print(f'测试集: {len(df_te):} 条, 坏率: {y_te.mean():.2%}')

## Step 3. 特征筛选

In [ ]:
sel = CompositeFeatureSelector([
    ('null',  NullSelector(threshold=0.5)),
    ('var',   VarianceSelector(threshold=0.0)),
    ('iv',    IVSelector(threshold=0.02, max_n_bins=8)),
    ('corr',  CorrSelector(threshold=0.85)),
])
sel.fit(X_tr_raw, y_tr)
X_tr_sel = sel.transform(X_tr_raw)
X_te_sel = sel.transform(X_te_raw)
sel_feats = X_tr_sel.columns.tolist()
print(f'筛选: {len(all_feats)} → {len(sel_feats)} 个特征')
print('保留特征:', sel_feats)

## Step 4. 最优分箱

In [ ]:
# 外部评分用单调分箱，其他用MDLP
external = [f for f in sel_feats if any(k in f for k in ['bj_','mobtech','bairong','xiaoniu','dxm','bcf'])]
behavior = [f for f in sel_feats if f not in external]

user_splits = {}  # 可在此指定业务切点
binner = OptimalBinning(method='mdlp', max_n_bins=8, user_splits=user_splits)
binner.fit(X_tr_sel, y_tr)

print('各特征分箱IV:')
for feat in sel_feats:
    bt = binner.bin_tables_.get(feat, pd.DataFrame())
    iv_val = bt['分档IV值'].sum() if '分档IV值' in bt.columns else 0
    print(f'  {feat:45s} IV={iv_val:.4f} 箱={len(bt)}')

In [ ]:
# 可视化关键特征分箱
fig, axes = plt.subplots(1, min(3,len(sel_feats)), figsize=(18,5))
for i, feat in enumerate(sel_feats[:3]):
    bt = binner.bin_tables_.get(feat, pd.DataFrame())
    if not bt.empty:
        bin_plot(bt, desc=feat, ax=axes[i] if len(sel_feats)>1 else axes)
plt.tight_layout(); plt.show()

## Step 5. WOE 编码

In [ ]:
woe = WOEEncoder(max_n_bins=8, method='mdlp')
X_woe_tr = woe.fit_transform(X_tr_sel, y_tr)
X_woe_te = woe.transform(X_te_sel)
print('WOE编码完成:', X_woe_tr.shape)
print(X_woe_tr.head(2).round(3))

## Step 6. 逻辑回归训练

In [ ]:
lr = LogisticRegression(C=0.1, max_iter=2000, calculate_stats=True)
lr.fit(X_woe_tr, y_tr)
y_prob_tr = lr.predict_proba(X_woe_tr)[:,1]
y_prob_te = lr.predict_proba(X_woe_te)[:,1]

print('=' * 50)
print(f'训练集  KS:{ks(y_tr,y_prob_tr):.4f}  AUC:{auc(y_tr,y_prob_tr):.4f}  Gini:{gini(y_tr,y_prob_tr):.4f}')
print(f'测试集  KS:{ks(y_te,y_prob_te):.4f}  AUC:{auc(y_te,y_prob_te):.4f}  Gini:{gini(y_te,y_prob_te):.4f}')
print('=' * 50)

In [ ]:
print('模型系数摘要:')
lr.summary()

In [ ]:
fig = ks_plot(y_prob_te, y_te, title='测试集 KS曲线'); plt.show()
fig = plot_weights(lr); plt.show()

## Step 7. 评分卡生成

In [ ]:
sc = ScoreCard(pdo=20, score=600, odds=1/19,
    lr_kwargs={'C':0.1,'max_iter':2000,'calculate_stats':True})
sc.fit(X_woe_tr, y_tr, woe_encoder=woe)
scores_tr = sc.predict(X_woe_tr)
scores_te = sc.predict(X_woe_te)
print('评分分布(测试集):')
print(pd.Series(scores_te).describe().round(1))

In [ ]:
print('评分卡表:')
sc.scorecard_table_.head(20)

In [ ]:
# 评分区间坏率分析
score_bins = pd.cut(scores_te,
    bins=[0,520,540,560,580,600,620,640,660,1000],
    labels=['≤520','520-540','540-560','560-580','580-600','600-620','620-640','640-660','>660'])
analysis = pd.DataFrame({'评分区间':score_bins,'坏标':y_te.values})
tbl = analysis.groupby('评分区间', observed=False)['坏标'].agg(['mean','count']).rename(columns={'mean':'坏率','count':'样本数'})
tbl['坏率'] = tbl['坏率'].map('{:.2%}'.format)
print(tbl)

## Step 8. 评分可视化

In [ ]:
fig = score_dist_plot(scores_te, y_te.values); plt.show()
fig = score_bin_plot(scores_te, y_te.values, n_bins=10); plt.show()
fig = score_ks_plot(datasets={'训练集':(y_tr.values,scores_tr),'测试集':(y_te.values,scores_te)}); plt.show()
fig = score_approval_badrate_curve(y_te.values, scores_te); plt.show()

## Step 9. 阈值策略分析

In [ ]:
fig = threshold_analysis_plot(y_te.values, y_prob_te, thresholds=np.linspace(0.05,0.5,20))
plt.show()

# 推荐拒绝率下的通过率/坏率
for reject_pct in [0.1, 0.2, 0.3, 0.4]:
    threshold = np.percentile(y_prob_te, (1-reject_pct)*100)
    approved = y_prob_te <= threshold
    approved_badrate = y_te.values[approved].mean()
    print(f'拒绝率{reject_pct:.0%}: 通过率={approved.mean():.1%}, 通过坏率={approved_badrate:.2%}')

## Step 10. 特征稳定性监控

In [ ]:
psi_result = batch_psi(X_tr_sel, X_te_sel, features=sel_feats)
psi_result['稳定性'] = psi_result['PSI'].apply(psi_rating)
print('特征PSI监控:')
print(psi_result)

score_psi = psi(scores_tr, scores_te, max_n_bins=10)
print(f'\n评分PSI: {score_psi:.4f}  ({psi_rating(score_psi)})')